#### CSCE 636 Project 3

**Name and UIN:** **Kaushik Sivakumar (936002746)**

In [1]:
#Importing Required Packages

import numpy as np
import pandas as pd
from itertools import combinations, product, permutations
from scipy.optimize import linprog
import json
import os
import math
from datetime import datetime
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Dense, LayerNormalization, MultiHeadAttention, Dropout
from keras_nlp.layers import TransformerEncoder

#Tensorflow packages
import tensorflow as tf
from tensorflow import keras
from keras import layers

from sklearn.model_selection import train_test_split
import joblib
from tamu_csce_636_project1 import Evaluator

From Project 1 and 2, the following cases exhibited relatively high log loss values:

1. G_9_4_5 – 7.8304
2. G_9_5_4 – 9.0352
3. G_9_6_3 – 9.6070
4. G_10_4_6 – 7.4608
5. G_10_5_5 – 9.6268
6. G_10_6_4 – 8.5646

Here, G_n_k_m represents the specific (n, k, m) configuration for each case.

To address the elevated log losses observed in these scenarios, the current project focuses on implementing a classification-based model tailored to these configurations, with the aim of achieving improved performance under the log loss metric.

In [2]:
data = joblib.load(r'C:\Users\kaush\OneDrive - Texas A&M University\CSCE 636\Project\Dataset Generation\m_height_dataset.pkl')
data

,n,k,m,result,P
0,10,5,2,266.264524,"[-53.30165452517249, 46.36747377721454, 12.267..."
1,9,4,2,140.163560,"[6.5178894459618135, -6.281201889271188, 99.66..."
2,10,4,3,167.692765,"[-67.02930762978451, 82.95217006264068, -6.583..."
3,9,4,3,565.010999,"[91.14110355866848, -15.260304068831033, -47.3..."
4,9,6,2,1170.076493,"[92.83943954357875, -97.82331459868368, 5.3277..."
...,...,...,...,...,...
10980475,9,5,3,507.532357,"[-48.61881142586215, 69.4632720461743, 95.5588..."
10980476,9,6,2,2386.744603,"[26.87847968002606, 80.10900455650034, -64.122..."
10980477,10,4,5,1021.338785,"[-16.864249906777644, 92.88468480118638, 68.66..."
10980478,9,5,2,314.973375,"[-36.35990965825941, 46.68958014963235, -33.42..."


#### Training DNNs:

In the regression DNNs developed in Project 1, the following input configurations resulted in notably high Log Loss and Mean Absolute Error (MAE) values:

1. n = 9, k = 4, m = 5
2. n = 9, k = 5, m = 4
3. n = 9, k = 6, m = 3
4. n = 10, k = 4 m = 6 
5. n = 10, k = 5, m = 5
6. n = 10, k = 6, m = 4

The dataset for these 6 scenarios contains extremely large values, which disproportionately impacts the MAE. To address the skewness and large dynamic range, the problem is reformulated as a classification task. This approach allows us to optimize for a logarithmic loss, by discretizing the output space into logarithmic bins, enabling a classification model to predict class probabilities rather than precise values.

#### Class Mapping Strategy
The m height values are divided into the below classes:

1–10: 10 classes (1, 2, ..., 10)

10–100: 10 classes (10, 20, ..., 100)

100–1000: 10 classes (100, 200, ..., 1000)

... and so on.

#### Case 4: n = 9, k = 4, m = 5

In [505]:
G_9_4_5 = data[(data.n == 9)&(data.k == 4)&(data.m == 5)]
G_9_4_5

,n,k,m,result,P
12,9,4,5,30757.772723,"[52.11322076992937, 38.4959990023647, -39.8952..."
37,9,4,5,4604.376616,"[58.63791554744731, 2.5978474417976116, 50.890..."
58,9,4,5,6286.660357,"[-25.074155792987114, 97.34825824177395, 15.49..."
66,9,4,5,35424.142797,"[69.0514840527911, -6.325573540465967, 31.7300..."
78,9,4,5,278821.621548,"[49.468689374838476, 72.84861001129707, 31.660..."
...,...,...,...,...,...
10980394,9,4,5,186083.799460,"[1.6760605702045694, -39.035333686385485, 2.83..."
10980400,9,4,5,13011.678996,"[58.60842564781498, 32.327101391128565, 28.064..."
10980406,9,4,5,35402.362115,"[0.43171584081433423, -40.172978518327355, 51...."
10980459,9,4,5,145493.283933,"[-39.367901455852895, 33.054984117695, -50.528..."


In [507]:
#Max and Min Value of the dataset
G_9_4_5[G_9_4_5.result.isin([G_9_4_5.result.max(), G_9_4_5.result.min()])]

,n,k,m,result,P
7490648,9,4,5,7.313329e+02,"[-51.68698743351488, 25.908166604371743, 27.02..."
8955995,9,4,5,1.586443e+09,"[10.411696891347447, -6.847100107333787, 8.205..."


In [582]:
'''
Checking the ranges of values of the m heights in the dataset.
'''

def range_of_m_height_values(G):
    for i in range(10):
        print(f"{10**i} - {10**(i+1)}")
        print(f"{len(G[(G.result >= 10**i)&(G.result <= 10**(i+1))])}")
        print()

#1-9
range_of_m_height_values(G_9_4_5)

1 - 10
0

10 - 100
0

100 - 1000
2

1000 - 10000
122402

10000 - 100000
279468

100000 - 1000000
50785

1000000 - 10000000
5502

10000000 - 100000000
538

100000000 - 1000000000
46

1000000000 - 10000000000
2



In [515]:
'''
Class Mapping Strategy

The m height values are divided into the below classes:

1–10: 10 classes (1, 2, ..., 10)

10–100: 10 classes (10, 20, ..., 100)

100–1000: 10 classes (100, 200, ..., 1000)

... and so on.
'''
def mapping_m_height_to_classes(start, G): 
    G_dict = dict()
    count = 0
    for i in range(10):
        if 10**i < start: #start - lowerbound for class mapping
            continue
        for j in range(10**i, 10**(i+1) ,10**i):
            if j > G.result.max():
                break
            G_dict[count] = j
            count+=1
    return G_dict
G_9_4_5_dict = mapping_m_height_to_classes(100, G_9_4_5)
G_dict = G_9_4_5_dict
G_9_4_5_dict

{0: 100,
 1: 200,
 2: 300,
 3: 400,
 4: 500,
 5: 600,
 6: 700,
 7: 800,
 8: 900,
 9: 1000,
 10: 2000,
 11: 3000,
 12: 4000,
 13: 5000,
 14: 6000,
 15: 7000,
 16: 8000,
 17: 9000,
 18: 10000,
 19: 20000,
 20: 30000,
 21: 40000,
 22: 50000,
 23: 60000,
 24: 70000,
 25: 80000,
 26: 90000,
 27: 100000,
 28: 200000,
 29: 300000,
 30: 400000,
 31: 500000,
 32: 600000,
 33: 700000,
 34: 800000,
 35: 900000,
 36: 1000000,
 37: 2000000,
 38: 3000000,
 39: 4000000,
 40: 5000000,
 41: 6000000,
 42: 7000000,
 43: 8000000,
 44: 9000000,
 45: 10000000,
 46: 20000000,
 47: 30000000,
 48: 40000000,
 49: 50000000,
 50: 60000000,
 51: 70000000,
 52: 80000000,
 53: 90000000,
 54: 100000000,
 55: 200000000,
 56: 300000000,
 57: 400000000,
 58: 500000000,
 59: 600000000,
 60: 700000000,
 61: 800000000,
 62: 900000000,
 63: 1000000000}

Training Data: 70%

Validation Data: 15%

Testing Data: 15%

In [517]:
def convert_df_to_tf_input(G): #G -> dataframe 
    X = []                     #returns X, y -> numpy arrays
    y = []
    for i in range(len(G)):
        #print(i)
        X.append([G.iloc[i].n] + [G.iloc[i].k] + [G.iloc[i].m] + G.iloc[i].P.tolist())
        y.append(G.iloc[i].result)

    X, y = np.array(X), np.array(y)
    
    print(f"Dimensions of Input: {X.shape}")
    print(f"Dimensions of Output: {y.shape}")

    X_train, X_val_test, y_train, y_val_test = train_test_split(X, y, test_size=0.3, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, test_size=0.5, random_state=42)
    
    return X_train, y_train, X_val, y_val, X_test, y_test

def convert_to_labels(x):
    
    return np.argmin(np.abs(x - np.array(list(G_dict.values()))))

G_9_4_5_X_train, G_9_4_5_y_train, G_9_4_5_X_val, G_9_4_5_y_val, G_9_4_5_X_test, G_9_4_5_y_test = convert_df_to_tf_input(G_9_4_5)
G_9_4_5_y_train_label = pd.DataFrame(G_9_4_5_y_train).loc[:,0].apply(convert_to_labels).to_numpy()
G_9_4_5_y_val_label = pd.DataFrame(G_9_4_5_y_val).loc[:,0].apply(convert_to_labels).to_numpy()
G_9_4_5_y_test_label = pd.DataFrame(G_9_4_5_y_test).loc[:,0].apply(convert_to_labels).to_numpy()

Dimensions of Input: (458745, 23)
Dimensions of Output: (458745,)


In [519]:
print(G_9_4_5_X_train.shape, G_9_4_5_X_val.shape, G_9_4_5_X_test.shape)
print(G_9_4_5_y_train.shape, G_9_4_5_y_val.shape, G_9_4_5_y_test.shape) 

(321121, 23) (68812, 23) (68812, 23)
(321121,) (68812,) (68812,)


In [521]:
'''
occurences of each class within the training data
'''
pd.DataFrame(G_9_4_5_y_train_label).value_counts()

0 
19    56014
18    53095
20    30779
21    18985
27    14745
22    13259
15    13212
14    13167
16    12643
17    12097
13    12022
28    10649
12     9608
23     9572
24     7233
25     5641
11     5586
26     4632
29     4564
30     2618
36     1702
31     1649
10     1504
37     1146
32     1146
33      876
34      655
35      531
38      487
39      292
40      191
45      159
41      123
46      102
42       82
43       74
9        60
47       44
44       41
48       38
49       24
54       18
53       11
50        9
51        9
55        9
52        6
58        3
63        3
57        2
56        2
62        1
6         1
Name: count, dtype: int64

In [529]:
def model(G_dict):
    model = keras.Sequential()
    model.add(layers.Dense(128, activation = 'relu'))
    layers.Dropout(0.2)
    model.add(layers.Dense(128, activation = 'relu'))
    layers.Dropout(0.2)
    model.add(layers.Dense(128, activation = 'relu'))
    layers.Dropout(0.2)
    model.add(layers.Dense(128, activation = 'relu'))
    layers.Dropout(0.2)
    model.add(layers.Dense(128, activation = 'relu'))
    layers.Dropout(0.2)
    model.add(layers.Dense(len(G_dict), activation = 'softmax'))
    return model
    
G_9_4_5_model = model(G_9_4_5_dict)

G_9_4_5_model.compile(optimizer = keras.optimizers.Adam(learning_rate=1e-5),
                      loss = 'sparse_categorical_crossentropy',
                      metrics = ['accuracy'],
                      )

G_9_4_5_callbacks = [
                    keras.callbacks.ModelCheckpoint("G_9_4_5.keras",
                                                    save_best_only=True),
                    ]

In [531]:
G_9_4_5_history = G_9_4_5_model.fit(G_9_4_5_X_train,
                                    G_9_4_5_y_train_label,
                                    validation_data = (G_9_4_5_X_val, G_9_4_5_y_val_label),
                                    callbacks=G_9_4_5_callbacks,
                                    epochs = 20)

Epoch 1/20
10036/10036 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.1066 - loss: 4.6107 - val_accuracy: 0.1645 - val_loss: 2.9307
Epoch 2/20
10036/10036 ━━━━━━━━━━━━━━━━━━━━ 19s 2ms/step - accuracy: 0.1663 - loss: 2.9082 - val_accuracy: 0.1727 - val_loss: 2.8550
Epoch 3/20
10036/10036 ━━━━━━━━━━━━━━━━━━━━ 19s 2ms/step - accuracy: 0.1728 - loss: 2.8518 - val_accuracy: 0.1707 - val_loss: 2.8372
Epoch 4/20
10036/10036 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - accuracy: 0.1726 - loss: 2.8390 - val_accuracy: 0.1724 - val_loss: 2.8304
Epoch 5/20
10036/10036 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.1726 - loss: 2.8318 - val_accuracy: 0.1723 - val_loss: 2.8269
Epoch 6/20
10036/10036 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.1739 - loss: 2.8267 - val_accuracy: 0.1729 - val_loss: 2.8250
Epoch 7/20
10036/10036 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.1766 - loss: 2.8221 - val_accuracy: 0.1730 - val_loss: 2.8247
Epoch 8/20
10036/10036 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 

In [559]:
G_9_4_5_loaded = keras.models.load_model("G_9_4_5.keras")

def log_loss(model, X_test, y_test, G_dict):
    prediction = model.predict(X_test)
    
    return ((np.log2(y_test) - np.log2(np.array([G_dict[np.argmax(element)] for element in prediction])))**2).mean()
    
def predicted_m_height(model, X_test, G_dict):
    prediction = model.predict(X_test)
    prediction = np.array([G_dict[np.argmax(element)] for element in prediction])
    __ = np.where(prediction < 1)[0]
    for i in range(len(prediction)):
        if i in __:
            prediction[i] = 1
    return prediction

print(f"Log Loss: {log_loss(G_9_4_5_loaded, G_9_4_5_X_test, G_9_4_5_y_test, G_9_4_5_dict)}")

print(f"Prediction: {predicted_m_height(G_9_4_5_loaded, G_9_4_5_X_test, G_9_4_5_dict)}")

2151/2151 ━━━━━━━━━━━━━━━━━━━━ 2s 957us/step
Log Loss: 3.8285625549213154
2151/2151 ━━━━━━━━━━━━━━━━━━━━ 2s 976us/step
Prediction: [20000 20000 20000 ... 20000 20000 20000]


The Log Loss has reduced from '7.830406' to '3.828'.

#### Case 7: n = 9, k = 5, m = 4

In [576]:
G_9_5_4 = data[(data.n == 9)&(data.k == 5)&(data.m == 4)]
G_9_5_4

,n,k,m,result,P
13,9,5,4,19863.375738,"[64.2903089407022, -97.17442013202066, -59.379..."
60,9,5,4,14566.802705,"[-61.8565628325334, 26.642727507250868, 92.964..."
65,9,5,4,23596.569064,"[-13.129949845005058, 17.296063329187376, 40.2..."
77,9,5,4,6754.208993,"[-31.65163050919037, 67.44104799563956, 96.316..."
113,9,5,4,15305.881540,"[-40.34015314899687, -1.6922378076438775, -10...."
...,...,...,...,...,...
10980396,9,5,4,9024.324825,"[-29.138518460683173, 84.46822691982604, -13.3..."
10980410,9,5,4,31532.786759,"[96.96057671646125, 8.694226406443079, 28.9712..."
10980418,9,5,4,7798.175920,"[-42.85443743031798, -82.69827664919276, -47.8..."
10980437,9,5,4,10902.160747,"[55.639330551302834, -68.53615156432743, 42.58..."


In [578]:
G_9_5_4[G_9_5_4.result.isin([G_9_5_4.result.max(), G_9_5_4.result.min()])]

,n,k,m,result,P
2585879,9,5,4,1.254703e+03,"[-27.975582672657723, 14.56148286711371, -31.7..."
9686551,9,5,4,2.078378e+09,"[-47.804544268764936, -92.74486107017788, 76.5..."


In [580]:
range_of_m_height_values(G_9_5_4)

1 - 10
0

10 - 100
0

100 - 1000
0

1000 - 10000
106455

10000 - 100000
404987

100000 - 1000000
87630

1000000 - 10000000
9567

10000000 - 100000000
971

100000000 - 1000000000
95

1000000000 - 10000000000
5



In [584]:
G_9_5_4_dict = mapping_m_height_to_classes(1000, G_9_5_4)
G_dict = G_9_5_4_dict
G_9_5_4_dict

{0: 1000,
 1: 2000,
 2: 3000,
 3: 4000,
 4: 5000,
 5: 6000,
 6: 7000,
 7: 8000,
 8: 9000,
 9: 10000,
 10: 20000,
 11: 30000,
 12: 40000,
 13: 50000,
 14: 60000,
 15: 70000,
 16: 80000,
 17: 90000,
 18: 100000,
 19: 200000,
 20: 300000,
 21: 400000,
 22: 500000,
 23: 600000,
 24: 700000,
 25: 800000,
 26: 900000,
 27: 1000000,
 28: 2000000,
 29: 3000000,
 30: 4000000,
 31: 5000000,
 32: 6000000,
 33: 7000000,
 34: 8000000,
 35: 9000000,
 36: 10000000,
 37: 20000000,
 38: 30000000,
 39: 40000000,
 40: 50000000,
 41: 60000000,
 42: 70000000,
 43: 80000000,
 44: 90000000,
 45: 100000000,
 46: 200000000,
 47: 300000000,
 48: 400000000,
 49: 500000000,
 50: 600000000,
 51: 700000000,
 52: 800000000,
 53: 900000000,
 54: 1000000000,
 55: 2000000000}

In [586]:
G_9_5_4_X_train, G_9_5_4_y_train, G_9_5_4_X_val, G_9_5_4_y_val, G_9_5_4_X_test, G_9_5_4_y_test = convert_df_to_tf_input(G_9_5_4)
G_9_5_4_y_train_label = pd.DataFrame(G_9_5_4_y_train).loc[:,0].apply(convert_to_labels).to_numpy()
G_9_5_4_y_val_label = pd.DataFrame(G_9_5_4_y_val).loc[:,0].apply(convert_to_labels).to_numpy()
G_9_5_4_y_test_label = pd.DataFrame(G_9_5_4_y_test).loc[:,0].apply(convert_to_labels).to_numpy()

Dimensions of Input: (609710, 23)
Dimensions of Output: (609710,)


In [587]:
print(G_9_5_4_X_train.shape, G_9_5_4_X_val.shape, G_9_5_4_X_test.shape)
print(G_9_5_4_y_train.shape, G_9_5_4_y_val.shape, G_9_5_4_y_test.shape) 

(426797, 23) (91456, 23) (91457, 23)
(426797,) (91456,) (91457,)


In [588]:
pd.DataFrame(G_9_5_4_y_train_label).value_counts()

0 
10    78238
9     66018
11    46542
12    30203
18    24924
13    21142
19    18278
14    15371
8     13413
7     13057
6     12619
15    12032
5     11315
16     9367
4      8946
20     8154
17     7530
3      5818
21     4530
22     2953
27     2851
2      2566
23     2022
28     1976
24     1524
25     1164
26      910
29      812
30      471
1       393
36      307
31      300
37      194
32      188
33      143
34      122
38       98
35       88
39       56
45       32
40       29
42       19
41       18
46       16
0        10
43        9
47        8
44        6
49        5
48        4
54        2
50        1
51        1
52        1
55        1
Name: count, dtype: int64

In [589]:
G_9_5_4_model = model(G_9_5_4_dict)

G_9_5_4_model.compile(optimizer = keras.optimizers.Adam(learning_rate=1e-5),
                      loss = 'sparse_categorical_crossentropy',
                      metrics = ['accuracy'],
                      )

G_9_5_4_callbacks = [
                    keras.callbacks.ModelCheckpoint("G_9_5_4.keras",
                                                    save_best_only=True),
                    ]

In [594]:
G_9_5_4_history = G_9_5_4_model.fit(G_9_5_4_X_train,
                                    G_9_5_4_y_train_label,
                                    validation_data = (G_9_5_4_X_val, G_9_5_4_y_val_label),
                                    callbacks=G_9_5_4_callbacks,
                                    epochs = 20)

Epoch 1/20
13338/13338 ━━━━━━━━━━━━━━━━━━━━ 27s 2ms/step - accuracy: 0.1137 - loss: 4.3664 - val_accuracy: 0.1702 - val_loss: 2.8695
Epoch 2/20
13338/13338 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - accuracy: 0.1710 - loss: 2.8530 - val_accuracy: 0.1796 - val_loss: 2.8164
Epoch 3/20
13338/13338 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - accuracy: 0.1787 - loss: 2.8130 - val_accuracy: 0.1820 - val_loss: 2.8042
Epoch 4/20
13338/13338 ━━━━━━━━━━━━━━━━━━━━ 26s 2ms/step - accuracy: 0.1822 - loss: 2.8036 - val_accuracy: 0.1839 - val_loss: 2.7994
Epoch 5/20
13338/13338 ━━━━━━━━━━━━━━━━━━━━ 26s 2ms/step - accuracy: 0.1823 - loss: 2.8014 - val_accuracy: 0.1846 - val_loss: 2.7972
Epoch 6/20
13338/13338 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - accuracy: 0.1833 - loss: 2.7972 - val_accuracy: 0.1842 - val_loss: 2.7955
Epoch 7/20
13338/13338 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - accuracy: 0.1823 - loss: 2.7964 - val_accuracy: 0.1847 - val_loss: 2.7948
Epoch 8/20
13338/13338 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - accuracy: 

In [597]:
G_9_5_4_loaded = keras.models.load_model("G_9_5_4.keras")

print(f"Log Loss: {log_loss(G_9_5_4_loaded, G_9_5_4_X_test, G_9_5_4_y_test, G_9_5_4_dict)}")

print(f"Prediction: {predicted_m_height(G_9_5_4_loaded, G_9_5_4_X_test, G_9_5_4_dict)}")

2859/2859 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
Log Loss: 3.9134260129811587
2859/2859 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
Prediction: [20000 20000 20000 ... 20000 20000 20000]


The Log Loss has reduced from '9.035178' to '3.913'.

#### Case 9: n = 9, k = 6, m = 3

In [601]:
G_9_6_3 = data[(data.n == 9)&(data.k == 6)&(data.m == 3)]
G_9_6_3

,n,k,m,result,P
27,9,6,3,20726.687921,"[36.76466888973516, -47.7424916770393, 91.3273..."
30,9,6,3,16373.452952,"[90.33840482367864, -50.28832508694984, -99.22..."
49,9,6,3,13088.839108,"[-82.28132026167475, 60.22660227469498, 54.759..."
51,9,6,3,16545.943681,"[-55.719764294548206, -46.407712419674674, 69...."
57,9,6,3,16341.609925,"[64.65536144986007, -12.145041318328865, -73.0..."
...,...,...,...,...,...
10980433,9,6,3,67353.628145,"[63.62991104733172, -86.82414644521016, 43.230..."
10980448,9,6,3,2555.680836,"[-60.002589671561566, -10.155454311416932, -16..."
10980461,9,6,3,46667.823832,"[83.28061005753327, 92.92215584267646, 12.5923..."
10980469,9,6,3,3679.188087,"[-55.43192057020572, -84.87278155156129, -94.6..."


In [603]:
G_9_6_3[G_9_6_3.result.isin([G_9_6_3.result.max(), G_9_6_3.result.min()])]

,n,k,m,result,P
8185801,9,6,3,4.828543e+09,"[-81.93852125712827, -29.149001012945718, -46...."
9532904,9,6,3,7.918200e+02,"[26.678512061202525, 23.298240260023135, 5.609..."


In [605]:
range_of_m_height_values(G_9_6_3)

1 - 10
0

10 - 100
0

100 - 1000
17

1000 - 10000
317537

10000 - 100000
505999

100000 - 1000000
81550

1000000 - 10000000
8575

10000000 - 100000000
821

100000000 - 1000000000
70

1000000000 - 10000000000
10



In [607]:
G_9_6_3_dict = mapping_m_height_to_classes(100, G_9_6_3)
G_dict = G_9_6_3_dict
G_9_6_3_dict

{0: 100,
 1: 200,
 2: 300,
 3: 400,
 4: 500,
 5: 600,
 6: 700,
 7: 800,
 8: 900,
 9: 1000,
 10: 2000,
 11: 3000,
 12: 4000,
 13: 5000,
 14: 6000,
 15: 7000,
 16: 8000,
 17: 9000,
 18: 10000,
 19: 20000,
 20: 30000,
 21: 40000,
 22: 50000,
 23: 60000,
 24: 70000,
 25: 80000,
 26: 90000,
 27: 100000,
 28: 200000,
 29: 300000,
 30: 400000,
 31: 500000,
 32: 600000,
 33: 700000,
 34: 800000,
 35: 900000,
 36: 1000000,
 37: 2000000,
 38: 3000000,
 39: 4000000,
 40: 5000000,
 41: 6000000,
 42: 7000000,
 43: 8000000,
 44: 9000000,
 45: 10000000,
 46: 20000000,
 47: 30000000,
 48: 40000000,
 49: 50000000,
 50: 60000000,
 51: 70000000,
 52: 80000000,
 53: 90000000,
 54: 100000000,
 55: 200000000,
 56: 300000000,
 57: 400000000,
 58: 500000000,
 59: 600000000,
 60: 700000000,
 61: 800000000,
 62: 900000000,
 63: 1000000000,
 64: 2000000000,
 65: 3000000000,
 66: 4000000000}

In [609]:
G_9_6_3_X_train, G_9_6_3_y_train, G_9_6_3_X_val, G_9_6_3_y_val, G_9_6_3_X_test, G_9_6_3_y_test = convert_df_to_tf_input(G_9_6_3)
G_9_6_3_y_train_label = pd.DataFrame(G_9_6_3_y_train).loc[:,0].apply(convert_to_labels).to_numpy()
G_9_6_3_y_val_label = pd.DataFrame(G_9_6_3_y_val).loc[:,0].apply(convert_to_labels).to_numpy()
G_9_6_3_y_test_label = pd.DataFrame(G_9_6_3_y_test).loc[:,0].apply(convert_to_labels).to_numpy()

Dimensions of Input: (914579, 21)
Dimensions of Output: (914579,)


In [610]:
print(G_9_6_3_X_train.shape, G_9_6_3_X_val.shape, G_9_6_3_X_test.shape)
print(G_9_6_3_y_train.shape, G_9_6_3_y_val.shape, G_9_6_3_y_test.shape) 

(640205, 21) (137187, 21) (137187, 21)
(640205,) (137187,) (137187,)


In [611]:
pd.DataFrame(G_9_6_3_y_train_label).value_counts()

0 
18    107369
19    103419
20     53942
13     33051
14     32962
21     32599
15     31334
12     29552
16     28722
17     26591
27     23762
22     21827
11     20391
28     16765
23     15812
24     11688
25      9251
29      7443
26      7362
10      7052
30      4199
31      2656
36      2477
32      1910
37      1821
33      1393
34      1003
35       791
38       726
39       453
9        381
40       301
45       257
46       179
41       177
42       145
43       109
44        88
47        62
48        51
49        24
55        23
54        19
50        14
51         8
52         8
8          7
56         5
7          5
53         4
63         4
57         2
62         2
64         2
58         1
59         1
60         1
65         1
66         1
Name: count, dtype: int64

In [612]:
G_9_6_3_model = model(G_9_6_3_dict)

G_9_6_3_model.compile(optimizer = keras.optimizers.Adam(learning_rate=1e-5),
                      loss = 'sparse_categorical_crossentropy',
                      metrics = ['accuracy'],
                      )

G_9_6_3_callbacks = [
                    keras.callbacks.ModelCheckpoint("G_9_6_3.keras",
                                                    save_best_only=True),
                    ]

In [617]:
G_9_6_3_history = G_9_6_3_model.fit(G_9_6_3_X_train,
                                    G_9_6_3_y_train_label,
                                    validation_data = (G_9_6_3_X_val, G_9_6_3_y_val_label),
                                    callbacks=G_9_6_3_callbacks,
                                    epochs = 20)

Epoch 1/20
20007/20007 ━━━━━━━━━━━━━━━━━━━━ 39s 2ms/step - accuracy: 0.1218 - loss: 3.9779 - val_accuracy: 0.1659 - val_loss: 2.8714
Epoch 2/20
20007/20007 ━━━━━━━━━━━━━━━━━━━━ 38s 2ms/step - accuracy: 0.1649 - loss: 2.8583 - val_accuracy: 0.1665 - val_loss: 2.8450
Epoch 3/20
20007/20007 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - accuracy: 0.1664 - loss: 2.8362 - val_accuracy: 0.1663 - val_loss: 2.8380
Epoch 4/20
20007/20007 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.1678 - loss: 2.8319 - val_accuracy: 0.1668 - val_loss: 2.8356
Epoch 5/20
20007/20007 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - accuracy: 0.1677 - loss: 2.8294 - val_accuracy: 0.1668 - val_loss: 2.8342
Epoch 6/20
20007/20007 ━━━━━━━━━━━━━━━━━━━━ 37s 2ms/step - accuracy: 0.1692 - loss: 2.8272 - val_accuracy: 0.1662 - val_loss: 2.8336
Epoch 7/20
20007/20007 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.1685 - loss: 2.8290 - val_accuracy: 0.1663 - val_loss: 2.8330
Epoch 8/20
20007/20007 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - accuracy: 

In [618]:
G_9_6_3_loaded = keras.models.load_model("G_9_6_3.keras")

print(f"Log Loss: {log_loss(G_9_6_3_loaded, G_9_6_3_X_test, G_9_6_3_y_test, G_9_6_3_dict)}")

print(f"Prediction: {predicted_m_height(G_9_6_3_loaded, G_9_6_3_X_test, G_9_6_3_dict)}")

4288/4288 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
Log Loss: 3.857455918701213
4288/4288 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
Prediction: [20000 20000 20000 ... 20000 20000 10000]


The Log Loss has reduced from '9.607' to '3.857'.

#### Case 14: n = 10, k = 4, m = 6

In [624]:
G_10_4_6 = data[(data.n == 10)&(data.k == 4)&(data.m == 6)]
G_10_4_6

,n,k,m,result,P
6,10,4,6,42822.483197,"[32.263006124132716, 59.34168874668569, 89.932..."
8,10,4,6,70381.722359,"[-28.346971695960633, -50.082233387203395, -9...."
26,10,4,6,106802.555920,"[65.0945731374133, 38.98952586366377, 24.48209..."
45,10,4,6,29554.279110,"[-45.5486588867863, -54.01813670334538, -52.45..."
255,10,4,6,15288.924294,"[-11.607028211190965, 13.147302821011749, -26...."
...,...,...,...,...,...
10980431,10,4,6,2347.112745,"[84.53083361376088, -16.427352005656275, 43.33..."
10980432,10,4,6,37391.469405,"[-19.021264290246293, 51.09635471235276, 67.07..."
10980445,10,4,6,99680.246057,"[-24.82667431662678, -73.05430522368076, -89.5..."
10980451,10,4,6,19121.482732,"[-10.134129028348056, 65.51758636709687, -23.8..."


In [626]:
G_10_4_6[G_10_4_6.result.isin([G_10_4_6.result.max(), G_10_4_6.result.min()])]

,n,k,m,result,P
3492532,10,4,6,1.590771e+09,"[82.45077377771523, -73.45116657476672, 63.809..."
6243009,10,4,6,1.594554e+03,"[35.10971411785758, -98.57497612440935, -25.43..."


In [628]:
range_of_m_height_values(G_10_4_6)

1 - 10
0

10 - 100
0

100 - 1000
0

1000 - 10000
38138

10000 - 100000
252160

100000 - 1000000
66321

1000000 - 10000000
7322

10000000 - 100000000
725

100000000 - 1000000000
67

1000000000 - 10000000000
2



In [630]:
G_10_4_6_dict = mapping_m_height_to_classes(1000, G_10_4_6)
G_dict = G_10_4_6_dict
G_10_4_6_dict

{0: 1000,
 1: 2000,
 2: 3000,
 3: 4000,
 4: 5000,
 5: 6000,
 6: 7000,
 7: 8000,
 8: 9000,
 9: 10000,
 10: 20000,
 11: 30000,
 12: 40000,
 13: 50000,
 14: 60000,
 15: 70000,
 16: 80000,
 17: 90000,
 18: 100000,
 19: 200000,
 20: 300000,
 21: 400000,
 22: 500000,
 23: 600000,
 24: 700000,
 25: 800000,
 26: 900000,
 27: 1000000,
 28: 2000000,
 29: 3000000,
 30: 4000000,
 31: 5000000,
 32: 6000000,
 33: 7000000,
 34: 8000000,
 35: 9000000,
 36: 10000000,
 37: 20000000,
 38: 30000000,
 39: 40000000,
 40: 50000000,
 41: 60000000,
 42: 70000000,
 43: 80000000,
 44: 90000000,
 45: 100000000,
 46: 200000000,
 47: 300000000,
 48: 400000000,
 49: 500000000,
 50: 600000000,
 51: 700000000,
 52: 800000000,
 53: 900000000,
 54: 1000000000}

In [632]:
G_10_4_6_X_train, G_10_4_6_y_train, G_10_4_6_X_val, G_10_4_6_y_val, G_10_4_6_X_test, G_10_4_6_y_test = convert_df_to_tf_input(G_10_4_6)
G_10_4_6_y_train_label = pd.DataFrame(G_10_4_6_y_train).loc[:,0].apply(convert_to_labels).to_numpy()
G_10_4_6_y_val_label = pd.DataFrame(G_10_4_6_y_val).loc[:,0].apply(convert_to_labels).to_numpy()
G_10_4_6_y_test_label = pd.DataFrame(G_10_4_6_y_test).loc[:,0].apply(convert_to_labels).to_numpy()

Dimensions of Input: (364735, 27)
Dimensions of Output: (364735,)


In [633]:
print(G_10_4_6_X_train.shape, G_10_4_6_X_val.shape, G_10_4_6_X_test.shape)
print(G_10_4_6_y_train.shape, G_10_4_6_y_val.shape, G_10_4_6_y_test.shape) 

(255314, 27) (54710, 27) (54711, 27)
(255314,) (54710,) (54711,)


In [634]:
pd.DataFrame(G_10_4_6_y_train_label).value_counts()

0 
10    46363
9     32555
11    30653
12    20523
18    18668
13    14825
19    13701
14    11013
15     8612
16     6842
20     6197
8      5753
17     5571
7      5341
6      4670
5      3676
21     3454
4      2538
22     2340
27     2216
23     1582
28     1541
3      1249
24     1208
25      846
26      683
29      652
2       397
30      350
31      243
36      221
32      159
37      133
33      128
38       76
34       73
35       72
40       33
39       33
1        29
45       21
46       16
42       10
41        9
43        9
47        8
44        6
49        6
48        5
52        2
54        2
50        1
Name: count, dtype: int64

In [638]:
G_10_4_6_model = model(G_10_4_6_dict)

G_10_4_6_model.compile(optimizer = keras.optimizers.Adam(learning_rate=1e-5),
                      loss = 'sparse_categorical_crossentropy',
                      metrics = ['accuracy'],
                      )

G_10_4_6_callbacks = [
                    keras.callbacks.ModelCheckpoint("G_10_4_6.keras",
                                                    save_best_only=True),
                    ]

In [640]:
G_10_4_6_history = G_10_4_6_model.fit(G_10_4_6_X_train,
                                     G_10_4_6_y_train_label,
                                     validation_data = (G_10_4_6_X_val, G_10_4_6_y_val_label),
                                     callbacks=G_10_4_6_callbacks,
                                     epochs = 20)

Epoch 1/20
7979/7979 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - accuracy: 0.1060 - loss: 5.2330 - val_accuracy: 0.1485 - val_loss: 2.9446
Epoch 2/20
7979/7979 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.1568 - loss: 2.8971 - val_accuracy: 0.1662 - val_loss: 2.8390
Epoch 3/20
7979/7979 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.1681 - loss: 2.8283 - val_accuracy: 0.1717 - val_loss: 2.8164
Epoch 4/20
7979/7979 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.1763 - loss: 2.8009 - val_accuracy: 0.1717 - val_loss: 2.8063
Epoch 5/20
7979/7979 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.1768 - loss: 2.7968 - val_accuracy: 0.1751 - val_loss: 2.8015
Epoch 6/20
7979/7979 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - accuracy: 0.1803 - loss: 2.7913 - val_accuracy: 0.1742 - val_loss: 2.7990
Epoch 7/20
7979/7979 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - accuracy: 0.1791 - loss: 2.7861 - val_accuracy: 0.1766 - val_loss: 2.7973
Epoch 8/20
7979/7979 ━━━━━━━━━━━━━━━━━━━━ 16s 2ms/step - accuracy: 0.1807 - loss: 2

In [641]:
G_10_4_6_loaded = keras.models.load_model("G_10_4_6.keras")

print(f"Log Loss: {log_loss(G_10_4_6_loaded, G_10_4_6_X_test, G_10_4_6_y_test, G_10_4_6_dict)}")

print(f"Prediction: {predicted_m_height(G_10_4_6_loaded, G_10_4_6_X_test, G_10_4_6_dict)}")

1710/1710 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
Log Loss: 4.48754112838414
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 2s 992us/step
Prediction: [20000 20000 10000 ... 20000 20000 20000]


The Log Loss has reduced from '7.46' to '4.487'.

#### Case 18: n = 10, k = 5, m = 5

In [644]:
G_10_5_5 = data[(data.n == 10)&(data.k == 5)&(data.m == 5)]
G_10_5_5

,n,k,m,result,P
134,10,5,5,9640.146544,"[86.00944570924926, 77.20836314076664, -67.182..."
135,10,5,5,39133.045116,"[34.055361835806394, 53.145476822870705, 60.62..."
146,10,5,5,46822.050499,"[-41.25843291529673, 22.340752765756022, -19.0..."
155,10,5,5,16986.719702,"[30.053172640908002, -83.19823621301492, 63.03..."
162,10,5,5,66771.425616,"[-5.409583939761234, 61.325515007078224, 24.47..."
...,...,...,...,...,...
10980350,10,5,5,20951.143575,"[-75.10121494421952, -48.96629663825818, 90.01..."
10980352,10,5,5,36717.860797,"[-0.537190718626988, -71.25868272416685, 1.229..."
10980374,10,5,5,80304.350109,"[16.219962173532878, 99.74533223873686, 78.899..."
10980438,10,5,5,275766.337343,"[18.02239094026335, 98.39178570523077, -12.808..."


In [646]:
G_10_5_5[G_10_5_5.result.isin([G_10_5_5.result.max(), G_10_5_5.result.min()])]

,n,k,m,result,P
4336016,10,5,5,1.741967e+10,"[19.156971782681808, -6.42828809935591, -36.35..."
5315288,10,5,5,3.280752e+03,"[-72.42069383900484, -74.20357460049513, 5.336..."


In [648]:
range_of_m_height_values(G_10_5_5)

1 - 10
0

10 - 100
0

100 - 1000
0

1000 - 10000
9430

10000 - 100000
297162

100000 - 1000000
133172

1000000 - 10000000
16307

10000000 - 100000000
1640

100000000 - 1000000000
163

1000000000 - 10000000000
7



In [650]:
G_10_5_5_dict = mapping_m_height_to_classes(1000, G_10_5_5)
G_dict = G_10_5_5_dict
G_10_5_5_dict

{0: 1000,
 1: 2000,
 2: 3000,
 3: 4000,
 4: 5000,
 5: 6000,
 6: 7000,
 7: 8000,
 8: 9000,
 9: 10000,
 10: 20000,
 11: 30000,
 12: 40000,
 13: 50000,
 14: 60000,
 15: 70000,
 16: 80000,
 17: 90000,
 18: 100000,
 19: 200000,
 20: 300000,
 21: 400000,
 22: 500000,
 23: 600000,
 24: 700000,
 25: 800000,
 26: 900000,
 27: 1000000,
 28: 2000000,
 29: 3000000,
 30: 4000000,
 31: 5000000,
 32: 6000000,
 33: 7000000,
 34: 8000000,
 35: 9000000,
 36: 10000000,
 37: 20000000,
 38: 30000000,
 39: 40000000,
 40: 50000000,
 41: 60000000,
 42: 70000000,
 43: 80000000,
 44: 90000000,
 45: 100000000,
 46: 200000000,
 47: 300000000,
 48: 400000000,
 49: 500000000,
 50: 600000000,
 51: 700000000,
 52: 800000000,
 53: 900000000,
 54: 1000000000,
 55: 2000000000,
 56: 3000000000,
 57: 4000000000,
 58: 5000000000,
 59: 6000000000,
 60: 7000000000,
 61: 8000000000,
 62: 9000000000}

In [652]:
G_10_5_5_X_train, G_10_5_5_y_train, G_10_5_5_X_val, G_10_5_5_y_val, G_10_5_5_X_test, G_10_5_5_y_test = convert_df_to_tf_input(G_10_5_5)
G_10_5_5_y_train_label = pd.DataFrame(G_10_5_5_y_train).loc[:,0].apply(convert_to_labels).to_numpy()
G_10_5_5_y_val_label = pd.DataFrame(G_10_5_5_y_val).loc[:,0].apply(convert_to_labels).to_numpy()
G_10_5_5_y_test_label = pd.DataFrame(G_10_5_5_y_test).loc[:,0].apply(convert_to_labels).to_numpy()

Dimensions of Input: (457883, 28)
Dimensions of Output: (457883,)


In [654]:
print(G_10_5_5_X_train.shape, G_10_5_5_X_val.shape, G_10_5_5_X_test.shape)
print(G_10_5_5_y_train.shape, G_10_5_5_y_val.shape, G_10_5_5_y_test.shape) 

(320518, 28) (68682, 28) (68683, 28)
(320518,) (68682,) (68683,)


In [656]:
pd.DataFrame(G_10_5_5_y_train_label).value_counts()

0 
10    42540
11    37123
18    35061
12    29106
19    27523
13    22927
14    18090
9     18082
15    14608
20    12834
16    12099
17    10030
21     7316
27     4956
22     4879
23     3382
28     3364
24     2586
8      2066
25     1956
26     1530
7      1503
29     1462
6       947
30      789
5       562
31      521
36      513
32      363
37      358
33      268
4       243
34      207
35      168
38      139
3        70
39       69
40       52
45       43
46       35
41       31
42       25
43       23
47       15
44       13
2         8
48        7
50        5
53        5
49        4
52        4
55        4
54        3
56        1
Name: count, dtype: int64

In [658]:
G_10_5_5_model = model(G_10_5_5_dict)

G_10_5_5_model.compile(optimizer = keras.optimizers.Adam(learning_rate=1e-5),
                      loss = 'sparse_categorical_crossentropy',
                      metrics = ['accuracy'],
                      )

G_10_5_5_callbacks = [
                    keras.callbacks.ModelCheckpoint("G_10_5_5.keras",
                                                    save_best_only=True),
                    ]

In [660]:
G_10_5_5_history = G_10_5_5_model.fit(G_10_5_5_X_train,
                                     G_10_5_5_y_train_label,
                                     validation_data = (G_10_5_5_X_val, G_10_5_5_y_val_label),
                                     callbacks=G_10_5_5_callbacks,
                                     epochs = 20)

Epoch 1/20
10017/10017 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.0742 - loss: 5.3276 - val_accuracy: 0.1064 - val_loss: 2.9420
Epoch 2/20
10017/10017 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.1141 - loss: 2.9064 - val_accuracy: 0.1176 - val_loss: 2.8581
Epoch 3/20
10017/10017 ━━━━━━━━━━━━━━━━━━━━ 19s 2ms/step - accuracy: 0.1212 - loss: 2.8480 - val_accuracy: 0.1221 - val_loss: 2.8375
Epoch 4/20
10017/10017 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.1255 - loss: 2.8266 - val_accuracy: 0.1244 - val_loss: 2.8292
Epoch 5/20
10017/10017 ━━━━━━━━━━━━━━━━━━━━ 19s 2ms/step - accuracy: 0.1276 - loss: 2.8219 - val_accuracy: 0.1277 - val_loss: 2.8250
Epoch 6/20
10017/10017 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.1301 - loss: 2.8174 - val_accuracy: 0.1269 - val_loss: 2.8227
Epoch 7/20
10017/10017 ━━━━━━━━━━━━━━━━━━━━ 19s 2ms/step - accuracy: 0.1297 - loss: 2.8166 - val_accuracy: 0.1262 - val_loss: 2.8216
Epoch 8/20
10017/10017 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 

In [661]:
G_10_5_5_loaded = keras.models.load_model("G_10_5_5.keras")

print(f"Log Loss: {log_loss(G_10_5_5_loaded, G_10_5_5_X_test, G_10_5_5_y_test, G_10_5_5_dict)}")

print(f"Prediction: {predicted_m_height(G_10_5_5_loaded, G_10_5_5_X_test, G_10_5_5_dict)}")

2147/2147 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
Log Loss: 6.338741249865535
2147/2147 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
Prediction: [20000 30000 20000 ... 20000 20000 20000]


The Log Loss has reduced from '9.627' to '6.339'.

#### Case 21: n = 10, k = 6, m = 4

In [666]:
G_10_6_4 = data[(data.n == 10)&(data.k == 6)&(data.m == 4)]
G_10_6_4

,n,k,m,result,P
19,10,6,4,16802.864235,"[-81.49775662810725, -7.6575088017866335, 98.5..."
41,10,6,4,63709.695569,"[-79.17563227699725, 57.47224398984798, 18.261..."
52,10,6,4,72410.539445,"[-92.44070253390242, 86.37360183420097, 89.496..."
53,10,6,4,40307.918252,"[67.78674564640576, 50.15771667480149, -36.726..."
71,10,6,4,32945.175993,"[6.877881235858155, 62.37212836948805, -30.680..."
...,...,...,...,...,...
10980388,10,6,4,6607.763424,"[-21.783349620622715, 44.56953391702601, 16.22..."
10980391,10,6,4,73701.193231,"[27.57991800476256, 94.708142818927, 24.820930..."
10980403,10,6,4,12338.127189,"[88.44136985567249, -64.49851740599809, -30.22..."
10980405,10,6,4,59535.795528,"[1.8290975831738194, 50.79535342302049, -1.641..."


In [668]:
G_10_6_4[G_10_6_4.result.isin([G_10_6_4.result.max(), G_10_6_4.result.min()])]

,n,k,m,result,P
5313311,10,6,4,3.619524e+09,"[-98.66207525981704, 97.09875291347169, -37.58..."
10286330,10,6,4,2.618899e+03,"[-46.42414460024862, -71.50702432604741, -19.9..."


In [670]:
range_of_m_height_values(G_10_6_4)

1 - 10
0

10 - 100
0

100 - 1000
0

1000 - 10000
14940

10000 - 100000
402687

100000 - 1000000
168407

1000000 - 10000000
20424

10000000 - 100000000
2077

100000000 - 1000000000
207

1000000000 - 10000000000
20



In [672]:
G_10_6_4_dict = mapping_m_height_to_classes(1000, G_10_6_4)
G_dict = G_10_6_4_dict
G_10_6_4_dict

{0: 1000,
 1: 2000,
 2: 3000,
 3: 4000,
 4: 5000,
 5: 6000,
 6: 7000,
 7: 8000,
 8: 9000,
 9: 10000,
 10: 20000,
 11: 30000,
 12: 40000,
 13: 50000,
 14: 60000,
 15: 70000,
 16: 80000,
 17: 90000,
 18: 100000,
 19: 200000,
 20: 300000,
 21: 400000,
 22: 500000,
 23: 600000,
 24: 700000,
 25: 800000,
 26: 900000,
 27: 1000000,
 28: 2000000,
 29: 3000000,
 30: 4000000,
 31: 5000000,
 32: 6000000,
 33: 7000000,
 34: 8000000,
 35: 9000000,
 36: 10000000,
 37: 20000000,
 38: 30000000,
 39: 40000000,
 40: 50000000,
 41: 60000000,
 42: 70000000,
 43: 80000000,
 44: 90000000,
 45: 100000000,
 46: 200000000,
 47: 300000000,
 48: 400000000,
 49: 500000000,
 50: 600000000,
 51: 700000000,
 52: 800000000,
 53: 900000000,
 54: 1000000000,
 55: 2000000000,
 56: 3000000000}

In [674]:
G_10_6_4_X_train, G_10_6_4_y_train, G_10_6_4_X_val, G_10_6_4_y_val, G_10_6_4_X_test, G_10_6_4_y_test = convert_df_to_tf_input(G_10_6_4)
G_10_6_4_y_train_label = pd.DataFrame(G_10_6_4_y_train).loc[:,0].apply(convert_to_labels).to_numpy()
G_10_6_4_y_val_label = pd.DataFrame(G_10_6_4_y_val).loc[:,0].apply(convert_to_labels).to_numpy()
G_10_6_4_y_test_label = pd.DataFrame(G_10_6_4_y_test).loc[:,0].apply(convert_to_labels).to_numpy()

Dimensions of Input: (608762, 27)
Dimensions of Output: (608762,)


In [675]:
pd.DataFrame(G_10_6_4_y_train_label).value_counts()

0 
10    60127
11    50599
18    44739
12    39033
19    35154
13    30135
9     27026
14    23650
15    18934
20    16207
16    15504
17    12993
21     9131
22     5958
27     5938
28     4311
23     4200
8      3119
24     3072
25     2442
7      2354
26     1925
29     1813
6      1615
30     1000
5       932
31      708
36      580
37      464
32      441
4       440
33      312
34      276
38      204
35      195
3       126
39      103
45       75
41       60
40       53
46       40
42       34
43       19
2        18
47       17
44       14
48       10
51        6
54        6
55        5
49        4
50        4
56        4
52        2
53        2
Name: count, dtype: int64

In [676]:
G_10_6_4_model = model(G_10_6_4_dict)

G_10_6_4_model.compile(optimizer = keras.optimizers.Adam(learning_rate=1e-5),
                      loss = 'sparse_categorical_crossentropy',
                      metrics = ['accuracy'],
                      )

G_10_6_4_callbacks = [
                    keras.callbacks.ModelCheckpoint("G_10_6_4.keras",
                                                    save_best_only=True),
                    ]

In [677]:
G_10_6_4_history = G_10_6_4_model.fit(G_10_6_4_X_train,
                                     G_10_6_4_y_train_label,
                                     validation_data = (G_10_6_4_X_val, G_10_6_4_y_val_label),
                                     callbacks=G_10_6_4_callbacks,
                                     epochs = 20)

Epoch 1/20
13317/13317 ━━━━━━━━━━━━━━━━━━━━ 26s 2ms/step - accuracy: 0.0878 - loss: 4.4226 - val_accuracy: 0.1190 - val_loss: 2.8880
Epoch 2/20
13317/13317 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - accuracy: 0.1272 - loss: 2.8628 - val_accuracy: 0.1313 - val_loss: 2.8292
Epoch 3/20
13317/13317 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.1339 - loss: 2.8200 - val_accuracy: 0.1363 - val_loss: 2.8157
Epoch 4/20
13317/13317 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.1376 - loss: 2.8140 - val_accuracy: 0.1367 - val_loss: 2.8105
Epoch 5/20
13317/13317 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.1389 - loss: 2.8049 - val_accuracy: 0.1392 - val_loss: 2.8082
Epoch 6/20
13317/13317 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - accuracy: 0.1391 - loss: 2.8057 - val_accuracy: 0.1386 - val_loss: 2.8066
Epoch 7/20
13317/13317 ━━━━━━━━━━━━━━━━━━━━ 25s 2ms/step - accuracy: 0.1395 - loss: 2.8026 - val_accuracy: 0.1391 - val_loss: 2.8059
Epoch 8/20
13317/13317 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 

In [678]:
G_10_6_4_loaded = keras.models.load_model("G_10_6_4.keras")

print(f"Log Loss: {log_loss(G_10_6_4_loaded, G_10_6_4_X_test, G_10_6_4_y_test, G_10_6_4_dict)}")

print(f"Prediction: {predicted_m_height(G_10_6_4_loaded, G_10_6_4_X_test, G_10_6_4_dict)}")

2854/2854 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step  
Log Loss: 6.30920276586841
2854/2854 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step   
Prediction: [20000 20000 20000 ... 30000 20000 20000]


The Log Loss has reduced from '8.565' to '6.309'.

#### Test Performance:

In [6]:
#Preparing Test dataset
from tamu_csce_636_project1 import Evaluator

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers

inputs = {}
outputs = {}
for n in [9,10]:
    for k in [4, 5, 6]:
        for m in range(2, n-k+1):
            temp = pd.read_csv(f'C://Users//kaush//OneDrive - Texas A&M University//CSCE 636//Project 2//Test_Dataset//G_{n}_{k}_{m}_test.csv')
            inputs[f"[{n},{k},{m}]"] = [element.reshape((k, n-k)) for element in temp.iloc[:,3:-1].to_numpy()]
            outputs[f"[{n},{k},{m}]"] = temp.result.to_list()

In [29]:

G_9_4_5_dict = {0: 100, 1: 200, 2: 300, 3: 400, 4: 500, 5: 600, 6: 700, 7: 800, 8: 900, 9: 1000, 10: 2000, 11: 3000, 12: 4000, 13: 5000, 
                14: 6000, 15: 7000, 16: 8000, 17: 9000, 18: 10000, 19: 20000, 20: 30000, 21: 40000, 22: 50000, 23: 60000, 24: 70000, 
                25: 80000, 26: 90000, 27: 100000, 28: 200000, 29: 300000, 30: 400000, 31: 500000, 32: 600000, 33: 700000, 34: 800000, 
                35: 900000, 36: 1000000, 37: 2000000, 38: 3000000, 39: 4000000, 40: 5000000, 41: 6000000, 42: 7000000, 43: 8000000, 44: 9000000, 
                45: 10000000, 46: 20000000, 47: 30000000, 48: 40000000, 49: 50000000, 50: 60000000, 51: 70000000, 52: 80000000, 53: 90000000, 
                54: 100000000, 55: 200000000, 56: 300000000, 57: 400000000, 58: 500000000, 59: 600000000, 60: 700000000, 61: 800000000, 62: 900000000, 
                63: 1000000000}

G_9_5_4_dict = {0: 1000, 1: 2000, 2: 3000, 3: 4000, 4: 5000, 5: 6000, 6: 7000, 7: 8000, 8: 9000, 9: 10000, 10: 20000, 11: 30000, 
                12: 40000, 13: 50000, 14: 60000, 15: 70000, 16: 80000, 17: 90000, 18: 100000, 19: 200000, 20: 300000, 21: 400000, 
                22: 500000, 23: 600000, 24: 700000, 25: 800000, 26: 900000, 27: 1000000, 28: 2000000, 29: 3000000, 30: 4000000, 31: 5000000, 
                32: 6000000, 33: 7000000, 34: 8000000, 35: 9000000, 36: 10000000, 37: 20000000, 38: 30000000, 39: 40000000, 40: 50000000, 
                41: 60000000, 42: 70000000, 43: 80000000, 44: 90000000, 45: 100000000, 46: 200000000, 47: 300000000, 48: 400000000, 
                49: 500000000, 50: 600000000, 51: 700000000, 52: 800000000, 53: 900000000, 54: 1000000000, 55: 2000000000}

G_9_6_3_dict = {0: 100, 1: 200, 2: 300, 3: 400, 4: 500, 5: 600, 6: 700, 7: 800, 8: 900, 9: 1000, 10: 2000, 11: 3000, 12: 4000, 13: 5000, 14: 6000, 
                15: 7000, 16: 8000, 17: 9000, 18: 10000, 19: 20000, 20: 30000, 21: 40000, 22: 50000, 23: 60000, 24: 70000, 25: 80000, 26: 90000, 
                27: 100000, 28: 200000, 29: 300000, 30: 400000, 31: 500000, 32: 600000, 33: 700000, 34: 800000, 35: 900000, 36: 1000000, 37: 2000000, 
                38: 3000000, 39: 4000000, 40: 5000000, 41: 6000000, 42: 7000000, 43: 8000000, 44: 9000000, 45: 10000000, 46: 20000000, 47: 30000000, 
                48: 40000000, 49: 50000000, 50: 60000000, 51: 70000000, 52: 80000000, 53: 90000000, 54: 100000000, 55: 200000000, 56: 300000000, 
                57: 400000000, 58: 500000000, 59: 600000000, 60: 700000000, 61: 800000000, 62: 900000000, 63: 1000000000, 64: 2000000000, 
                65: 3000000000, 66: 4000000000}

G_10_4_6_dict = {0: 1000, 1: 2000, 2: 3000, 3: 4000, 4: 5000, 5: 6000, 6: 7000, 7: 8000, 8: 9000, 9: 10000, 10: 20000, 11: 30000, 12: 40000, 
                 13: 50000, 14: 60000, 15: 70000, 16: 80000, 17: 90000, 18: 100000, 19: 200000, 20: 300000, 21: 400000, 22: 500000, 23: 600000, 
                 24: 700000, 25: 800000, 26: 900000, 27: 1000000, 28: 2000000, 29: 3000000, 30: 4000000, 31: 5000000, 32: 6000000, 33: 7000000, 
                 34: 8000000, 35: 9000000, 36: 10000000, 37: 20000000, 38: 30000000, 39: 40000000, 40: 50000000, 41: 60000000, 42: 70000000, 
                 43: 80000000, 44: 90000000, 45: 100000000, 46: 200000000, 47: 300000000, 48: 400000000, 49: 500000000, 50: 600000000, 
                 51: 700000000, 52: 800000000, 53: 900000000, 54: 1000000000}

G_10_5_5_dict = {0: 1000, 1: 2000, 2: 3000, 3: 4000, 4: 5000, 5: 6000, 6: 7000, 7: 8000, 8: 9000, 9: 10000, 10: 20000, 11: 30000, 12: 40000, 13: 50000, 
                 14: 60000, 15: 70000, 16: 80000, 17: 90000, 18: 100000, 19: 200000, 20: 300000, 21: 400000, 22: 500000, 23: 600000, 24: 700000, 
                 25: 800000, 26: 900000, 27: 1000000, 28: 2000000, 29: 3000000, 30: 4000000, 31: 5000000, 32: 6000000, 33: 7000000, 34: 8000000, 
                 35: 9000000, 36: 10000000, 37: 20000000, 38: 30000000, 39: 40000000, 40: 50000000, 41: 60000000, 42: 70000000, 43: 80000000, 
                 44: 90000000, 45: 100000000, 46: 200000000, 47: 300000000, 48: 400000000, 49: 500000000, 50: 600000000, 51: 700000000, 
                 52: 800000000, 53: 900000000, 54: 1000000000, 55: 2000000000, 56: 3000000000, 57: 4000000000, 58: 5000000000, 59: 6000000000, 
                 60: 7000000000, 61: 8000000000, 62: 9000000000}

G_10_6_4_dict = {0: 1000, 1: 2000, 2: 3000, 3: 4000, 4: 5000, 5: 6000, 6: 7000, 7: 8000, 8: 9000, 9: 10000, 10: 20000, 11: 30000, 12: 40000, 13: 50000, 
                 14: 60000, 15: 70000, 16: 80000, 17: 90000, 18: 100000, 19: 200000, 20: 300000, 21: 400000, 22: 500000, 23: 600000, 24: 700000, 
                 25: 800000, 26: 900000, 27: 1000000, 28: 2000000, 29: 3000000, 30: 4000000, 31: 5000000, 32: 6000000, 33: 7000000, 34: 8000000, 
                 35: 9000000, 36: 10000000, 37: 20000000, 38: 30000000, 39: 40000000, 40: 50000000, 41: 60000000, 42: 70000000, 43: 80000000, 
                 44: 90000000, 45: 100000000, 46: 200000000, 47: 300000000, 48: 400000000, 49: 500000000, 50: 600000000, 51: 700000000, 52: 800000000, 
                 53: 900000000, 54: 1000000000, 55: 2000000000, 56: 3000000000}


def DNN_for_m_height(n, k, m, p_list):  
    
    n, k, m = int(n), int(k), int(m)
    
    X = [[n] + [k] + [m] + p.flatten().tolist() for p in p_list]
    X = np.array(X)

    model = keras.models.load_model(f"G_{n}_{k}_{m}.keras")

    if (n, k, m) in ((9, 4, 5), (9, 5, 4), (9, 6, 3), (10, 4, 6), (10, 5, 5), (10, 6, 4)):
        prediction = model.predict(X)
        prediction = np.array( eval(f"[G_{n}_{k}_{m}_dict[np.argmax(element)] for element in prediction]") )
    else:
        prediction = model.predict(X).flatten()
    
    __ = np.where(prediction < 1)[0]
    for i in range(len(prediction)):
        if i in __:
            prediction[i] = 1
            
    return prediction.tolist()

evaluator = Evaluator(
    first_name="Your Name",
    last_name="Your Name",
    email="email@tamu.edu",
    print=False,
)

σ = evaluator.eval(
    inputs=inputs,
    outputs=outputs,
    func=DNN_for_m_height,
)

test_performance = pd.DataFrame([list(dict(σ).values()), np.array([len(inputs[list(inputs.keys())[i]]) for i in range(0, 21)])]).T
test_performance.columns = ['Log_Loss', 'no_of_samples']
test_performance

2143/2143 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step
2150/2150 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step
2139/2139 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step
2151/2151 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
2863/2863 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step
2859/2859 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step
2859/2859 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
4290/4290 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step
4288/4288 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
1711/1711 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step
1717/1717 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step
1720/1720 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step
1716/1716 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
2149/2149 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step
2152/2152 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step
2144/2144 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step
2147/2147 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
2860/2860 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step
2861/2861 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step
2854/2854 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step  


,Log_Loss,no_of_samples
0,0.126815,68557.0
1,0.236053,68796.0
2,1.074123,68424.0
3,3.828563,68812.0
4,0.220841,91590.0
5,0.996560,91470.0
6,3.913426,91457.0
7,0.554307,137268.0
8,3.857456,137187.0
9,0.910560,54747.0


In [31]:
print(f"Test Log Loss: {np.dot(test_performance.Log_Loss, test_performance.no_of_samples)/test_performance.no_of_samples.sum()}")

Test Log Loss: 1.8482476846658082


The Overall Log Loss value has been reduced from '3.139' (project 1 value) to '**1.84**'.

#### Code that needs to be executed:

__Please run all the cells from here onwards.__

The 'DNN_for_m_height' function takes the following inputs:
- n (int)
- k (int)
- m (int) 
- p_list (list of 2D numpy arrays)

and returns

- a list of scalars (m height values) for each matrix in p_list

In [ ]:
!pip install numpy
!pip install tensorflow
!pip install tamu_csce_636_project1

In [1045]:
from tamu_csce_636_project1 import Evaluator

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers


G_9_4_5_dict = {0: 100, 1: 200, 2: 300, 3: 400, 4: 500, 5: 600, 6: 700, 7: 800, 8: 900, 9: 1000, 10: 2000, 11: 3000, 12: 4000, 13: 5000, 
                14: 6000, 15: 7000, 16: 8000, 17: 9000, 18: 10000, 19: 20000, 20: 30000, 21: 40000, 22: 50000, 23: 60000, 24: 70000, 
                25: 80000, 26: 90000, 27: 100000, 28: 200000, 29: 300000, 30: 400000, 31: 500000, 32: 600000, 33: 700000, 34: 800000, 
                35: 900000, 36: 1000000, 37: 2000000, 38: 3000000, 39: 4000000, 40: 5000000, 41: 6000000, 42: 7000000, 43: 8000000, 44: 9000000, 
                45: 10000000, 46: 20000000, 47: 30000000, 48: 40000000, 49: 50000000, 50: 60000000, 51: 70000000, 52: 80000000, 53: 90000000, 
                54: 100000000, 55: 200000000, 56: 300000000, 57: 400000000, 58: 500000000, 59: 600000000, 60: 700000000, 61: 800000000, 62: 900000000, 
                63: 1000000000}

G_9_5_4_dict = {0: 1000, 1: 2000, 2: 3000, 3: 4000, 4: 5000, 5: 6000, 6: 7000, 7: 8000, 8: 9000, 9: 10000, 10: 20000, 11: 30000, 
                12: 40000, 13: 50000, 14: 60000, 15: 70000, 16: 80000, 17: 90000, 18: 100000, 19: 200000, 20: 300000, 21: 400000, 
                22: 500000, 23: 600000, 24: 700000, 25: 800000, 26: 900000, 27: 1000000, 28: 2000000, 29: 3000000, 30: 4000000, 31: 5000000, 
                32: 6000000, 33: 7000000, 34: 8000000, 35: 9000000, 36: 10000000, 37: 20000000, 38: 30000000, 39: 40000000, 40: 50000000, 
                41: 60000000, 42: 70000000, 43: 80000000, 44: 90000000, 45: 100000000, 46: 200000000, 47: 300000000, 48: 400000000, 
                49: 500000000, 50: 600000000, 51: 700000000, 52: 800000000, 53: 900000000, 54: 1000000000, 55: 2000000000}

G_9_6_3_dict = {0: 100, 1: 200, 2: 300, 3: 400, 4: 500, 5: 600, 6: 700, 7: 800, 8: 900, 9: 1000, 10: 2000, 11: 3000, 12: 4000, 13: 5000, 14: 6000, 
                15: 7000, 16: 8000, 17: 9000, 18: 10000, 19: 20000, 20: 30000, 21: 40000, 22: 50000, 23: 60000, 24: 70000, 25: 80000, 26: 90000, 
                27: 100000, 28: 200000, 29: 300000, 30: 400000, 31: 500000, 32: 600000, 33: 700000, 34: 800000, 35: 900000, 36: 1000000, 37: 2000000, 
                38: 3000000, 39: 4000000, 40: 5000000, 41: 6000000, 42: 7000000, 43: 8000000, 44: 9000000, 45: 10000000, 46: 20000000, 47: 30000000, 
                48: 40000000, 49: 50000000, 50: 60000000, 51: 70000000, 52: 80000000, 53: 90000000, 54: 100000000, 55: 200000000, 56: 300000000, 
                57: 400000000, 58: 500000000, 59: 600000000, 60: 700000000, 61: 800000000, 62: 900000000, 63: 1000000000, 64: 2000000000, 
                65: 3000000000, 66: 4000000000}

G_10_4_6_dict = {0: 1000, 1: 2000, 2: 3000, 3: 4000, 4: 5000, 5: 6000, 6: 7000, 7: 8000, 8: 9000, 9: 10000, 10: 20000, 11: 30000, 12: 40000, 
                 13: 50000, 14: 60000, 15: 70000, 16: 80000, 17: 90000, 18: 100000, 19: 200000, 20: 300000, 21: 400000, 22: 500000, 23: 600000, 
                 24: 700000, 25: 800000, 26: 900000, 27: 1000000, 28: 2000000, 29: 3000000, 30: 4000000, 31: 5000000, 32: 6000000, 33: 7000000, 
                 34: 8000000, 35: 9000000, 36: 10000000, 37: 20000000, 38: 30000000, 39: 40000000, 40: 50000000, 41: 60000000, 42: 70000000, 
                 43: 80000000, 44: 90000000, 45: 100000000, 46: 200000000, 47: 300000000, 48: 400000000, 49: 500000000, 50: 600000000, 
                 51: 700000000, 52: 800000000, 53: 900000000, 54: 1000000000}

G_10_5_5_dict = {0: 1000, 1: 2000, 2: 3000, 3: 4000, 4: 5000, 5: 6000, 6: 7000, 7: 8000, 8: 9000, 9: 10000, 10: 20000, 11: 30000, 12: 40000, 13: 50000, 
                 14: 60000, 15: 70000, 16: 80000, 17: 90000, 18: 100000, 19: 200000, 20: 300000, 21: 400000, 22: 500000, 23: 600000, 24: 700000, 
                 25: 800000, 26: 900000, 27: 1000000, 28: 2000000, 29: 3000000, 30: 4000000, 31: 5000000, 32: 6000000, 33: 7000000, 34: 8000000, 
                 35: 9000000, 36: 10000000, 37: 20000000, 38: 30000000, 39: 40000000, 40: 50000000, 41: 60000000, 42: 70000000, 43: 80000000, 
                 44: 90000000, 45: 100000000, 46: 200000000, 47: 300000000, 48: 400000000, 49: 500000000, 50: 600000000, 51: 700000000, 
                 52: 800000000, 53: 900000000, 54: 1000000000, 55: 2000000000, 56: 3000000000, 57: 4000000000, 58: 5000000000, 59: 6000000000, 
                 60: 7000000000, 61: 8000000000, 62: 9000000000}

G_10_6_4_dict = {0: 1000, 1: 2000, 2: 3000, 3: 4000, 4: 5000, 5: 6000, 6: 7000, 7: 8000, 8: 9000, 9: 10000, 10: 20000, 11: 30000, 12: 40000, 13: 50000, 
                 14: 60000, 15: 70000, 16: 80000, 17: 90000, 18: 100000, 19: 200000, 20: 300000, 21: 400000, 22: 500000, 23: 600000, 24: 700000, 
                 25: 800000, 26: 900000, 27: 1000000, 28: 2000000, 29: 3000000, 30: 4000000, 31: 5000000, 32: 6000000, 33: 7000000, 34: 8000000, 
                 35: 9000000, 36: 10000000, 37: 20000000, 38: 30000000, 39: 40000000, 40: 50000000, 41: 60000000, 42: 70000000, 43: 80000000, 
                 44: 90000000, 45: 100000000, 46: 200000000, 47: 300000000, 48: 400000000, 49: 500000000, 50: 600000000, 51: 700000000, 52: 800000000, 
                 53: 900000000, 54: 1000000000, 55: 2000000000, 56: 3000000000}

def DNN_for_m_height(n, k, m, p_list):  
    
    """
    Parameters:
        n (int)
        k (int)
        m (int)
        p_list (list): a list of P matrices, each matrix will be a numpy array

    Returns:
        a list of scalars, e.g. the m-height value, one for each matrix
    """
    
    n, k, m = int(n), int(k), int(m)
    
    X = [[n] + [k] + [m] + p.flatten().tolist() for p in p_list]
    X = np.array(X)

    model = keras.models.load_model(f"G_{n}_{k}_{m}.keras")

    if (n, k, m) in ((9, 4, 5), (9, 5, 4), (9, 6, 3), (10, 4, 6), (10, 5, 5), (10, 6, 4)):
        prediction = model.predict(X)
        prediction = np.array( eval(f"[G_{n}_{k}_{m}_dict[np.argmax(element)] for element in prediction]") )
    else:
        prediction = model.predict(X).flatten()
    
    __ = np.where(prediction < 1)[0]
    for i in range(len(prediction)):
        if i in __:
            prediction[i] = 1
            
    return prediction.tolist()

In [ ]:
"""
Enter the test dataset here 

evaluator = Evaluator(
    first_name="Your Name",
    last_name="Your Name",
    email="email@tamu.edu",
    print=False,
)

σ = evaluator.eval(
    inputs=inputs,
    outputs=outputs,
    func=DNN_for_m_height,
)
"""